# 🧬 Dark Manifold V8 — Complete Integration

## What V8 Combines

| Source | Feature | Why It Matters |
|--------|---------|----------------|
| **V3** | Hamiltonian dynamics | Energy conservation |
| **V3** | Thermodynamic loss | ΔG × v ≤ 0 |
| **V3** | Conservation loss | Mass balance |
| **V3** | W_reg gene regulation | Knockout prediction |
| **V5** | Km diversity loss | Differentiated kinetics |
| **V7** | Environment coupling | Realistic cell-media interaction |
| **NEW** | Trajectory MSE | Supervised ground truth |
| **NEW** | ODE-generated targets | Physics-based training data |

## Key Insight

V7 failed because it had no trajectory supervision — just heuristic penalties.
V8 generates physics-based trajectories with an ODE solver, then trains to match them.

In [ ]:
#@title 1️⃣ Setup & Load Data
from google.colab import files
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
from tqdm import tqdm
import matplotlib.pyplot as plt

print("Upload syn3a_real_data.pkl:")
uploaded = files.upload()

with open('syn3a_real_data.pkl', 'rb') as f:
    raw_data = pickle.load(f)

print("✅ Data loaded!")

In [ ]:
#@title 2️⃣ Parse Real Biochemistry Data

dfs = raw_data['all_dataframes']
rxns_df = pd.DataFrame(dfs['reconstruction.xlsx:Reactions'])
mets_df = pd.DataFrame(dfs['reconstruction.xlsx:Metabolites'])

metabolites = mets_df['Metabolite ID'].tolist()
met_names = mets_df['Metabolite name'].tolist()
met_to_idx = {m: i for i, m in enumerate(metabolites)}
name_to_idx = {n.lower(): i for i, n in enumerate(met_names)}

# Parse genes from GPR rules
genes = set()
gene_to_rxns = {}
for j, gpr in enumerate(rxns_df['GPR rule'].tolist()):
    if pd.isna(gpr):
        continue
    gene_ids = re.findall(r'MMSYN1_\d+', str(gpr))
    for g in gene_ids:
        genes.add(g)
        if g not in gene_to_rxns:
            gene_to_rxns[g] = []
        gene_to_rxns[g].append(j)

genes = sorted(list(genes))
gene_to_idx = {g: i for i, g in enumerate(genes)}

n_genes = len(genes)
n_mets = len(metabolites)
n_rxns = len(rxns_df)

# Build stoichiometry matrix
S = np.zeros((n_mets, n_rxns))
for j, row in rxns_df.iterrows():
    equation = row['Reaction equation'].lower()
    for i, name in enumerate(met_names):
        if name.lower() in equation:
            left = equation.split('-->')[0] if '-->' in equation else equation.split('<==>')[0]
            S[i, j] = -1 if name.lower() in left else +1

# Gene-reaction matrix
G = np.zeros((n_genes, n_rxns))
for g, rxn_indices in gene_to_rxns.items():
    g_idx = gene_to_idx[g]
    for r_idx in rxn_indices:
        G[g_idx, r_idx] = 1

# Key metabolite indices
def find_met_idx(keyword):
    for i, n in enumerate(met_names):
        if keyword in n.lower():
            return i
    return 0

atp_idx = find_met_idx('atp')
adp_idx = find_met_idx('adp')
amp_idx = find_met_idx('amp')
glucose_idx = find_met_idx('d-glucose')
pyruvate_idx = find_met_idx('pyruvate')
lactate_idx = find_met_idx('lactate')
nad_idx = find_met_idx('nad+')
nadh_idx = find_met_idx('nadh')

print(f"Data: {n_genes} genes, {n_mets} metabolites, {n_rxns} reactions")
print(f"Stoichiometry non-zero: {np.count_nonzero(S)} ({100*np.count_nonzero(S)/S.size:.2f}%)")
print(f"\nKey indices: ATP={atp_idx}, ADP={adp_idx}, Glucose={glucose_idx}, Lactate={lactate_idx}")

In [ ]:
#@title 3️⃣ Environment Definition (from V7)

ENV_VARS = [
    'glucose_ext',    # External glucose (mM)
    'lactate_ext',    # External lactate (mM)
    'oxygen',         # Oxygen availability (0-1)
    'amino_acids',    # Amino acid pool (mM)
    'ph',             # pH (6.0-8.0)
    'temperature',    # Temperature factor (0.8-1.2)
]
n_env = len(ENV_VARS)

RICH_MEDIUM = {
    'glucose_ext': 10.0,
    'lactate_ext': 0.0,
    'oxygen': 1.0,
    'amino_acids': 5.0,
    'ph': 7.0,
    'temperature': 1.0,
}

def env_to_tensor(env_dict, device='cpu'):
    return torch.tensor([env_dict[k] for k in ENV_VARS], dtype=torch.float32, device=device)

print(f"Environment variables: {n_env}")

In [ ]:
#@title 4️⃣ Physics-Based Trajectory Generator (NEW)

class PhysicsTrajectoryGenerator:
    """
    Generate training trajectories using real stoichiometry + Michaelis-Menten kinetics.
    This provides SUPERVISED targets for the neural network.
    """
    
    def __init__(self, S, G, n_genes, n_mets, n_rxns, device='cpu'):
        self.S = torch.tensor(S, dtype=torch.float32, device=device)
        self.G = torch.tensor(G, dtype=torch.float32, device=device)
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.n_rxns = n_rxns
        self.device = device
        
        # Substrate mask (which metabolites are substrates for each reaction)
        self.substrate_mask = (self.S < 0).float()
        
        # Realistic Km values (varied per reaction)
        self.Km = torch.rand(n_rxns, device=device) * 2.0 + 0.1  # 0.1-2.1 mM
        
        # Vmax scaling
        self.Vmax_base = torch.rand(n_rxns, device=device) * 0.5 + 0.1  # 0.1-0.6
    
    def simulate(self, n_steps=50, dt=0.01, batch_size=1):
        """
        Generate trajectory using explicit Euler + Michaelis-Menten.
        Returns trajectory tensors for supervised training.
        """
        # Initial conditions
        gene_expr = torch.rand(batch_size, self.n_genes, device=self.device) * 1.5 + 0.5
        
        # Metabolites with realistic initial concentrations
        met_conc = torch.rand(batch_size, self.n_mets, device=self.device) * 1.0 + 0.5
        met_conc[:, atp_idx] = 4.0 + torch.rand(batch_size, device=self.device) * 0.5
        met_conc[:, adp_idx] = 0.5 + torch.rand(batch_size, device=self.device) * 0.2
        met_conc[:, glucose_idx] = 2.0 + torch.rand(batch_size, device=self.device) * 1.0
        
        # Environment
        env = torch.zeros(batch_size, n_env, device=self.device)
        env[:, 0] = 10.0 + torch.rand(batch_size, device=self.device) * 5.0  # glucose_ext
        env[:, 1] = 0.0  # lactate_ext
        env[:, 2] = 1.0  # oxygen
        env[:, 3] = 5.0  # amino_acids
        env[:, 4] = 7.0  # pH
        env[:, 5] = 1.0  # temperature
        
        # Trajectories
        gene_traj = [gene_expr.clone()]
        met_traj = [met_conc.clone()]
        env_traj = [env.clone()]
        flux_traj = []
        
        for step in range(n_steps):
            # Enzyme activity from genes via G matrix
            enzyme_activity = gene_expr @ self.G  # (B, n_rxns)
            Vmax = enzyme_activity * self.Vmax_base.unsqueeze(0)
            
            # Substrate concentrations for each reaction
            n_substrates = self.substrate_mask.sum(dim=0).clamp(min=1)
            substrate_conc = (met_conc @ self.substrate_mask) / n_substrates + 0.001
            
            # Michaelis-Menten kinetics
            flux = Vmax * substrate_conc / (self.Km.unsqueeze(0) + substrate_conc)
            
            # Add small noise for realism
            flux = flux + 0.005 * torch.randn_like(flux)
            flux = flux.clamp(min=0)
            
            # dM/dt = S @ flux
            dM_dt = flux @ self.S.T
            
            # Glucose uptake from environment
            glucose_ext = env[:, 0]
            glucose_uptake = 0.1 * glucose_ext / (0.5 + glucose_ext)
            glucose_uptake = glucose_uptake.clamp(max=glucose_ext * 0.1)  # Max 10% per step
            
            # Lactate export
            lactate_int = met_conc[:, lactate_idx]
            lactate_export = 0.05 * lactate_int / (1.0 + lactate_int)
            
            # Update metabolites (no inplace)
            met_conc = met_conc + dt * dM_dt
            glucose_add = torch.zeros_like(met_conc)
            glucose_add[:, glucose_idx] = glucose_uptake
            lactate_sub = torch.zeros_like(met_conc)
            lactate_sub[:, lactate_idx] = -lactate_export
            met_conc = (met_conc + glucose_add + lactate_sub).clamp(min=0.001)
            
            # Update environment (no inplace)
            env_0_new = (env[:, 0] - glucose_uptake).clamp(min=0)
            env_1_new = env[:, 1] + lactate_export
            env_4_new = 7.0 - 0.1 * env_1_new.clamp(max=5.0)
            env = torch.stack([env_0_new, env_1_new, env[:, 2], env[:, 3], env_4_new, env[:, 5]], dim=-1)  # pH drops with lactate
            
            # Small gene expression drift
            gene_expr = gene_expr + 0.001 * torch.randn_like(gene_expr)
            gene_expr = gene_expr.clamp(0.1, 2.0)
            
            gene_traj.append(gene_expr.clone())
            met_traj.append(met_conc.clone())
            env_traj.append(env.clone())
            flux_traj.append(flux.clone())
        
        return {
            'gene_trajectory': torch.stack(gene_traj, dim=1),  # (B, T+1, n_genes)
            'met_trajectory': torch.stack(met_traj, dim=1),    # (B, T+1, n_mets)
            'env_trajectory': torch.stack(env_traj, dim=1),    # (B, T+1, n_env)
            'flux_trajectory': torch.stack(flux_traj, dim=1),  # (B, T, n_rxns)
        }

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

generator = PhysicsTrajectoryGenerator(S, G, n_genes, n_mets, n_rxns, device)

# Test
test_traj = generator.simulate(n_steps=20, batch_size=2)
print(f"\nTest trajectory shapes:")
print(f"  Genes: {test_traj['gene_trajectory'].shape}")
print(f"  Metabolites: {test_traj['met_trajectory'].shape}")
print(f"  Environment: {test_traj['env_trajectory'].shape}")
print(f"  Flux: {test_traj['flux_trajectory'].shape}")

In [ ]:
#@title 5️⃣ Dark Manifold V8 Model

class DarkManifoldV8(nn.Module):
    """
    Complete integration:
    - V3 Hamiltonian structure
    - V3 W_reg gene regulation
    - V5 Learnable Km with diversity
    - V7 Environment coupling
    - NEW: Trained on physics-based trajectories
    """
    
    def __init__(self, n_genes, n_mets, n_rxns, n_env, S, G, hidden_dim=256):
        super().__init__()
        
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.n_rxns = n_rxns
        self.n_env = n_env
        
        # Fixed biochemistry
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        self.register_buffer('substrate_mask', (torch.tensor(S) < 0).float())
        
        # ============ FROM V3: Gene Regulation ============
        # W_reg: gene-gene regulatory interactions
        self.W_reg = nn.Parameter(torch.randn(n_genes, n_genes) * 0.01)
        
        # ============ FROM V7: Environment Sensing ============
        self.env_sensor = nn.Sequential(
            nn.Linear(n_env, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, n_genes),
            nn.Tanh(),
        )
        
        # Transport parameters
        self.log_glucose_Km = nn.Parameter(torch.tensor(0.0))
        self.log_glucose_Vmax = nn.Parameter(torch.tensor(-1.0))
        self.log_lactate_Km = nn.Parameter(torch.tensor(0.0))
        self.log_lactate_Vmax = nn.Parameter(torch.tensor(-1.0))
        
        # ============ Gene → Enzyme ============
        self.gene_encoder = nn.Sequential(
            nn.Linear(n_genes, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_rxns),
            nn.Softplus(),
        )
        
        # ============ FROM V5: Learnable Km ============
        self.log_Km = nn.Parameter(torch.randn(n_rxns) * 0.5)  # Start varied
        self.log_tau = nn.Parameter(torch.zeros(n_rxns))
        
        # ============ Metabolite encoder ============
        self.met_encoder = nn.Sequential(
            nn.Linear(n_mets, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        
        # ============ FROM V3: Hamiltonian ============
        # Energy function H(q, p) where q=metabolites, p=fluxes
        self.hamiltonian_net = nn.Sequential(
            nn.Linear(n_mets + n_rxns, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
        )
        
        # ============ Flux predictor ============
        self.flux_net = nn.Sequential(
            nn.Linear(hidden_dim + n_rxns, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_rxns),
        )
        
        # ============ Growth predictor ============
        self.growth_net = nn.Sequential(
            nn.Linear(n_mets + n_env, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid(),
        )
    
    @property
    def Km(self):
        return torch.exp(self.log_Km).clamp(0.01, 100.0)
    
    @property
    def tau(self):
        return torch.exp(self.log_tau).clamp(0.1, 10.0)
    
    def compute_hamiltonian(self, met_conc, flux):
        """Compute energy of the system."""
        state = torch.cat([met_conc, flux], dim=-1)
        return self.hamiltonian_net(state).squeeze(-1)
    
    def forward(self, gene_expr, met_conc, env_state, dt=0.01):
        """
        One step of dynamics.
        """
        batch_size = gene_expr.shape[0]
        
        # ===== 1. GENE REGULATION (V3) =====
        # W_reg captures gene-gene interactions
        reg_signal = torch.tanh(gene_expr @ self.W_reg.T)
        
        # ===== 2. ENVIRONMENT SENSING (V7) =====
        env_signal = self.env_sensor(env_state)
        
        # Combined regulation
        regulated_genes = gene_expr * (1.0 + 0.3 * reg_signal + 0.2 * env_signal)
        regulated_genes = regulated_genes.clamp(min=0.1)
        
        # ===== 3. ENZYME KINETICS =====
        Vmax = self.gene_encoder(regulated_genes)  # (B, n_rxns)
        
        # Substrate concentrations
        n_substrates = self.substrate_mask.sum(dim=0).clamp(min=1)
        substrate_conc = (met_conc @ self.substrate_mask) / n_substrates + 0.001
        
        # Michaelis-Menten (V5 style)
        mm_term = substrate_conc / (self.Km.unsqueeze(0) + substrate_conc)
        
        # Metabolite-modulated flux
        met_hidden = self.met_encoder(met_conc)
        flux_input = torch.cat([met_hidden, Vmax], dim=-1)
        flux_mod = torch.sigmoid(self.flux_net(flux_input))
        
        # Combined flux
        flux = Vmax * mm_term * flux_mod * self.tau.unsqueeze(0)
        
        # ===== 4. HAMILTONIAN ENERGY (V3) =====
        H = self.compute_hamiltonian(met_conc, flux)
        
        # ===== 5. STOICHIOMETRY =====
        dM_dt = flux @ self.S.T
        
        # ===== 6. TRANSPORT (V7) =====
        glucose_ext = env_state[:, 0]
        glucose_Km = torch.exp(self.log_glucose_Km).clamp(0.01, 10.0)
        glucose_Vmax = torch.exp(self.log_glucose_Vmax).clamp(0.01, 2.0)
        glucose_uptake = glucose_Vmax * glucose_ext / (glucose_Km + glucose_ext)
        glucose_uptake = glucose_uptake.clamp(max=glucose_ext * 0.1)
        
        lactate_int = met_conc[:, lactate_idx]
        lactate_Km = torch.exp(self.log_lactate_Km).clamp(0.01, 10.0)
        lactate_Vmax = torch.exp(self.log_lactate_Vmax).clamp(0.01, 2.0)
        lactate_export = lactate_Vmax * lactate_int / (lactate_Km + lactate_int)
        lactate_export = lactate_export.clamp(max=lactate_int * 0.1)
        
        # ===== 7. UPDATE STATES =====
        # Build met_next without inplace operations
        met_next = met_conc + dt * dM_dt
        # Add transport effects via scatter_add-like operation
        glucose_delta = torch.zeros_like(met_next)
        glucose_delta[:, glucose_idx] = glucose_uptake
        lactate_delta = torch.zeros_like(met_next)
        lactate_delta[:, lactate_idx] = -lactate_export
        met_next = (met_next + glucose_delta + lactate_delta).clamp(min=0.001)
        
        # Build env_next without inplace operations
        env_0 = (env_state[:, 0] - glucose_uptake).clamp(min=0)
        env_1 = env_state[:, 1] + lactate_export
        env_4 = 7.0 - 0.1 * env_1.clamp(max=5.0)
        env_next = torch.stack([
            env_0,
            env_1,
            env_state[:, 2],
            env_state[:, 3],
            env_4,
            env_state[:, 5],
        ], dim=-1)
        
        # ===== 8. GROWTH =====
        growth_input = torch.cat([met_next, env_next], dim=-1)
        growth_rate = self.growth_net(growth_input).squeeze(-1)
        
        return {
            'met_next': met_next,
            'env_next': env_next,
            'flux': flux,
            'hamiltonian': H,
            'glucose_uptake': glucose_uptake,
            'lactate_export': lactate_export,
            'growth_rate': growth_rate,
            'regulated_genes': regulated_genes,
            'dM_dt': dM_dt,
        }
    
    def rollout(self, gene_expr, met_init, env_init, n_steps, dt=0.01):
        """Multi-step trajectory."""
        met_traj = [met_init.clone()]
        env_traj = [env_init.clone()]
        flux_traj = []
        H_traj = []
        
        met = met_init.clone()
        env = env_init.clone()
        for _ in range(n_steps):
            out = self.forward(gene_expr, met, env, dt)
            met = out['met_next']
            env = out['env_next']
            met_traj.append(met)
            env_traj.append(env)
            flux_traj.append(out['flux'])
            H_traj.append(out['hamiltonian'])
        
        return {
            'met_trajectory': torch.stack(met_traj, dim=1),
            'env_trajectory': torch.stack(env_traj, dim=1),
            'flux_trajectory': torch.stack(flux_traj, dim=1),
            'hamiltonian_trajectory': torch.stack(H_traj, dim=1),
        }


model = DarkManifoldV8(
    n_genes=n_genes,
    n_mets=n_mets,
    n_rxns=n_rxns,
    n_env=n_env,
    S=S,
    G=G,
    hidden_dim=256,
).to(device)

print(f"\nDarkManifoldV8 (Complete Integration):")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  W_reg shape: {model.W_reg.shape}")
print(f"  Km shape: {model.Km.shape}")

In [ ]:
#@title 6️⃣ Complete Loss Function

class V8Loss(nn.Module):
    """
    Complete loss combining all components:
    - Trajectory MSE (supervised)
    - Hamiltonian conservation (V3)
    - Thermodynamic consistency (V3)
    - Mass conservation (V3)
    - Km diversity (V5)
    - Environment dynamics (V7)
    - W_reg sparsity (NEW)
    """
    
    def __init__(self, atp_idx, adp_idx, amp_idx, glucose_idx, lactate_idx):
        super().__init__()
        self.atp_idx = atp_idx
        self.adp_idx = adp_idx
        self.amp_idx = amp_idx
        self.glucose_idx = glucose_idx
        self.lactate_idx = lactate_idx
    
    def trajectory_loss(self, pred_met, true_met):
        """Supervised trajectory matching (CRITICAL!)."""
        return F.mse_loss(pred_met, true_met)
    
    def hamiltonian_loss(self, H_traj):
        """Energy should be approximately conserved (V3)."""
        # dH/dt should be small
        dH = H_traj[:, 1:] - H_traj[:, :-1]
        return (dH ** 2).mean()
    
    def thermodynamic_loss(self, flux, dM_dt, met_conc):
        """
        Thermodynamic consistency: ΔG × v ≤ 0 (V3).
        Reactions should flow in thermodynamically favorable direction.
        """
        # Approximate ΔG from concentration ratios
        # For simplicity: penalize flux in "wrong" direction
        product_conc = F.relu(dM_dt)  # Products
        substrate_conc = F.relu(-dM_dt)  # Substrates
        
        # Penalize if making products when product concentration is already high
        penalty = F.relu(flux * (product_conc.mean(dim=-1, keepdim=True) - 2.0))
        return penalty.mean()
    
    def conservation_loss(self, met_traj):
        """Mass conservation: total mass shouldn't explode (V3)."""
        total_mass = met_traj.sum(dim=-1)  # (B, T)
        mass_change = (total_mass[:, 1:] - total_mass[:, :-1]).abs()
        return mass_change.mean()
    
    def km_diversity_loss(self, log_Km):
        """Km values should be diverse, not all the same (V5)."""
        Km = torch.exp(log_Km)
        # Penalize low variance
        km_std = Km.std()
        return F.relu(0.3 - km_std)  # Want std > 0.3
    
    def environment_loss(self, env_traj, glucose_uptake, lactate_export):
        """Environment should change realistically (V7)."""
        # Glucose should deplete
        glucose_start = env_traj[:, 0, 0]
        glucose_end = env_traj[:, -1, 0]
        depletion = glucose_start - glucose_end
        depletion_penalty = F.relu(0.5 - depletion).mean()
        
        # Should have positive uptake
        uptake_penalty = F.relu(0.01 - glucose_uptake).mean()
        
        return depletion_penalty + uptake_penalty
    
    def wreg_sparsity_loss(self, W_reg):
        """W_reg should be sparse but not dead (NEW)."""
        # L1 for sparsity
        l1 = W_reg.abs().mean()
        
        # But penalize if ALL are near zero
        max_val = W_reg.abs().max()
        dead_penalty = F.relu(0.01 - max_val)
        
        return 0.01 * l1 + 10.0 * dead_penalty
    
    def energy_charge_loss(self, met_traj):
        """Energy charge should stay healthy."""
        atp = met_traj[:, :, self.atp_idx]
        adp = met_traj[:, :, self.adp_idx]
        amp = met_traj[:, :, self.amp_idx]
        
        ec = (atp + 0.5 * adp) / (atp + adp + amp + 1e-6)
        ec_penalty = F.relu(0.7 - ec).mean()
        
        return ec_penalty
    
    def forward(self, pred_traj, true_traj, H_traj, flux, dM_dt, 
                met_conc, env_traj, log_Km, W_reg, glucose_uptake, lactate_export):
        """
        Combined loss with all components.
        """
        # Primary: trajectory matching (supervised!)
        loss_traj = self.trajectory_loss(pred_traj, true_traj)
        
        # Physics (V3)
        loss_H = self.hamiltonian_loss(H_traj)
        loss_thermo = self.thermodynamic_loss(flux, dM_dt, met_conc)
        loss_conserve = self.conservation_loss(pred_traj)
        loss_ec = self.energy_charge_loss(pred_traj)
        
        # Kinetics (V5)
        loss_km = self.km_diversity_loss(log_Km)
        
        # Environment (V7)
        loss_env = self.environment_loss(env_traj, glucose_uptake, lactate_export)
        
        # Regulation (NEW)
        loss_wreg = self.wreg_sparsity_loss(W_reg)
        
        # Weighted combination
        total = (
            10.0 * loss_traj +      # PRIMARY: supervised trajectory
            1.0 * loss_H +          # Hamiltonian conservation
            0.5 * loss_thermo +     # Thermodynamic direction
            0.5 * loss_conserve +   # Mass conservation
            1.0 * loss_ec +         # Energy charge
            2.0 * loss_km +         # Km diversity
            1.0 * loss_env +        # Environment dynamics
            1.0 * loss_wreg         # W_reg not dead
        )
        
        return total, {
            'trajectory': loss_traj.item(),
            'hamiltonian': loss_H.item(),
            'thermodynamic': loss_thermo.item(),
            'conservation': loss_conserve.item(),
            'energy_charge': loss_ec.item(),
            'km_diversity': loss_km.item(),
            'environment': loss_env.item(),
            'wreg': loss_wreg.item(),
        }


loss_fn = V8Loss(atp_idx, adp_idx, amp_idx, glucose_idx, lactate_idx)
print("✅ V8Loss defined with all components")

In [ ]:
#@title 7️⃣ Train V8

# Training config (V3-style: smaller batch, lower LR)
n_epochs = 500
batch_size = 8  # Smaller for stability
n_steps = 30
lr = 3e-4  # Lower LR like V3

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

losses = []
loss_components = {k: [] for k in ['trajectory', 'hamiltonian', 'thermodynamic', 
                                    'conservation', 'energy_charge', 'km_diversity',
                                    'environment', 'wreg']}

print(f"Training V8 for {n_epochs} epochs...")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: {lr}")
print(f"  Trajectory steps: {n_steps}")
print()

for epoch in tqdm(range(n_epochs)):
    model.train()
    
    # Generate physics-based target trajectories (detached - no grad needed)
    with torch.no_grad():
        target = generator.simulate(n_steps=n_steps, batch_size=batch_size)
    
    # Clone inputs to avoid inplace modification issues
    gene_expr = target['gene_trajectory'][:, 0].clone()
    met_init = target['met_trajectory'][:, 0].clone()
    env_init = target['env_trajectory'][:, 0].clone()
    true_met_traj = target['met_trajectory'].clone()
    
    # Model rollout
    pred = model.rollout(gene_expr, met_init.clone(), env_init.clone(), n_steps)
    
    # Get intermediate values for physics losses (fresh forward pass)
    out = model.forward(gene_expr.clone(), met_init.clone(), env_init.clone())
    
    # Compute loss
    loss, components = loss_fn(
        pred_traj=pred['met_trajectory'],
        true_traj=true_met_traj,
        H_traj=pred['hamiltonian_trajectory'],
        flux=out['flux'],
        dM_dt=out['dM_dt'],
        met_conc=met_init,
        env_traj=pred['env_trajectory'],
        log_Km=model.log_Km,
        W_reg=model.W_reg,
        glucose_uptake=out['glucose_uptake'],
        lactate_export=out['lactate_export'],
    )
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    losses.append(loss.item())
    for k, v in components.items():
        loss_components[k].append(v)
    
    if (epoch + 1) % 50 == 0:
        print(f"\nEpoch {epoch+1:3d}: Total={loss.item():.4f}")
        print(f"  Traj={components['trajectory']:.4f}, H={components['hamiltonian']:.4f}, "
              f"Km={components['km_diversity']:.4f}, Wreg={components['wreg']:.4f}")

print(f"\n✅ Training complete! Final loss: {losses[-1]:.4f}")

In [ ]:
#@title 8️⃣ Evaluate: Trajectory Prediction

model.eval()

# Generate test trajectory
test_target = generator.simulate(n_steps=50, batch_size=1)
gene_test = test_target['gene_trajectory'][:, 0]
met_test_init = test_target['met_trajectory'][:, 0]
env_test_init = test_target['env_trajectory'][:, 0]
true_traj = test_target['met_trajectory'][0].cpu().numpy()

# Predict
with torch.no_grad():
    pred = model.rollout(gene_test, met_test_init, env_test_init, n_steps=50)

pred_traj = pred['met_trajectory'][0].cpu().numpy()
pred_env = pred['env_trajectory'][0].cpu().numpy()

# Metrics
corr = np.corrcoef(true_traj.flatten(), pred_traj.flatten())[0, 1]
mse = np.mean((true_traj - pred_traj)**2)

print(f"Trajectory Prediction:")
print(f"  Correlation: {corr:.4f}")
print(f"  MSE: {mse:.6f}")

# Plot metabolites
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

met_indices = [atp_idx, adp_idx, glucose_idx, lactate_idx, 0, 1]
met_labels = ['ATP', 'ADP', 'Glucose', 'Lactate', 'Met 0', 'Met 1']

for i, (ax, idx, label) in enumerate(zip(axes.flatten(), met_indices, met_labels)):
    ax.plot(true_traj[:, idx], 'b-', linewidth=2, label='True (ODE)')
    ax.plot(pred_traj[:, idx], 'r--', linewidth=2, label='Predicted (V8)')
    ax.set_title(label)
    ax.set_xlabel('Time step')
    ax.set_ylabel('mM')
    ax.legend()

plt.suptitle(f'V8 Trajectory Prediction (Corr={corr:.3f})', fontsize=14)
plt.tight_layout()
plt.savefig('trajectory_v8.png', dpi=150)
plt.show()

# Plot environment
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].plot(pred_env[:, 0], 'b-', linewidth=2)
axes[0].set_title('External Glucose')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('mM')

axes[1].plot(pred_env[:, 1], 'orange', linewidth=2)
axes[1].set_title('External Lactate')
axes[1].set_xlabel('Time')

axes[2].plot(pred_env[:, 4], 'purple', linewidth=2)
axes[2].set_title('pH')
axes[2].set_xlabel('Time')

plt.suptitle('V8 Environment Dynamics', fontsize=14)
plt.tight_layout()
plt.savefig('environment_v8.png', dpi=150)
plt.show()

In [ ]:
#@title 9️⃣ Check Learned Parameters

print("="*60)
print("LEARNED PARAMETERS")
print("="*60)

# Km values
Km = model.Km.detach().cpu().numpy()
print(f"\nKm values:")
print(f"  Range: [{Km.min():.3f}, {Km.max():.3f}] mM")
print(f"  Mean: {Km.mean():.3f} mM")
print(f"  Std: {Km.std():.3f} mM")
print(f"  Diversity: {'✅ GOOD' if Km.std() > 0.3 else '⚠️ LOW'}")

# W_reg
W_reg = model.W_reg.detach().cpu().numpy()
print(f"\nW_reg (gene regulation):")
print(f"  Shape: {W_reg.shape}")
print(f"  Max abs: {np.abs(W_reg).max():.4f}")
print(f"  Mean abs: {np.abs(W_reg).mean():.4f}")
print(f"  Non-zero (>0.01): {(np.abs(W_reg) > 0.01).sum()}")
print(f"  Status: {'✅ ACTIVE' if np.abs(W_reg).max() > 0.05 else '⚠️ WEAK'}")

# Transport
print(f"\nTransport parameters:")
print(f"  Glucose Km: {torch.exp(model.log_glucose_Km).item():.3f} mM")
print(f"  Glucose Vmax: {torch.exp(model.log_glucose_Vmax).item():.3f} mM/step")
print(f"  Lactate Km: {torch.exp(model.log_lactate_Km).item():.3f} mM")
print(f"  Lactate Vmax: {torch.exp(model.log_lactate_Vmax).item():.3f} mM/step")

# Plot Km distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(Km, bins=30, edgecolor='black')
plt.xlabel('Km (mM)')
plt.ylabel('Count')
plt.title('Km Distribution')

plt.subplot(1, 2, 2)
plt.imshow(np.abs(W_reg[:30, :30]), cmap='hot')
plt.colorbar(label='|W_reg|')
plt.title('W_reg (first 30x30)')
plt.xlabel('Gene j')
plt.ylabel('Gene i')

plt.tight_layout()
plt.savefig('parameters_v8.png', dpi=150)
plt.show()

In [ ]:
#@title 🔟 Gene Knockout Analysis

print("="*60)
print("KNOCKOUT ANALYSIS")
print("="*60)

# Initial conditions
gene_wt = torch.ones(1, n_genes, device=device)
met_init = torch.ones(1, n_mets, device=device) * 0.5
met_init[0, atp_idx] = 4.0
met_init[0, adp_idx] = 0.5
met_init[0, glucose_idx] = 2.0

env_init = torch.zeros(1, n_env, device=device)
env_init[0, 0] = 5.0  # Limited glucose
env_init[0, 2] = 1.0  # oxygen
env_init[0, 3] = 3.0  # amino acids
env_init[0, 4] = 7.0  # pH
env_init[0, 5] = 1.0  # temperature

# Wildtype
with torch.no_grad():
    wt_result = model.rollout(gene_wt, met_init, env_init, n_steps=100)

wt_atp = wt_result['met_trajectory'][0, -1, atp_idx].item()
wt_glucose_consumed = env_init[0, 0].item() - wt_result['env_trajectory'][0, -1, 0].item()

print(f"Wildtype:")
print(f"  Final ATP: {wt_atp:.2f} mM")
print(f"  Glucose consumed: {wt_glucose_consumed:.2f} mM")

# Test knockouts
ko_results = []
for i in tqdm(range(n_genes), desc="Knockouts"):
    gene_ko = gene_wt.clone()
    gene_ko[0, i] = 0.0
    
    with torch.no_grad():
        ko_result = model.rollout(gene_ko, met_init.clone(), env_init.clone(), n_steps=100)
    
    ko_atp = ko_result['met_trajectory'][0, -1, atp_idx].item()
    ko_glucose = env_init[0, 0].item() - ko_result['env_trajectory'][0, -1, 0].item()
    
    # Essential if ATP drops >50% or glucose consumption drops >70%
    is_essential = (ko_atp < 0.5 * wt_atp) or (ko_glucose < 0.3 * wt_glucose_consumed)
    
    ko_results.append({
        'gene': genes[i],
        'atp': ko_atp,
        'atp_ratio': ko_atp / (wt_atp + 1e-6),
        'glucose': ko_glucose,
        'glucose_ratio': ko_glucose / (wt_glucose_consumed + 1e-6),
        'is_essential': is_essential,
    })

# Summary
essential = [r for r in ko_results if r['is_essential']]
print(f"\nResults:")
print(f"  Essential: {len(essential)}/{n_genes} ({100*len(essential)/n_genes:.1f}%)")

# Most essential
sorted_ko = sorted(ko_results, key=lambda x: x['atp_ratio'])
print(f"\nTop 10 most essential:")
for r in sorted_ko[:10]:
    status = "❌" if r['is_essential'] else "✓"
    print(f"  {status} {r['gene']}: ATP={r['atp_ratio']:.0%}, Glucose={r['glucose_ratio']:.0%}")

In [ ]:
#@title 1️⃣1️⃣ Loss Curves

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Total loss
axes[0, 0].semilogy(losses)
axes[0, 0].set_title('Total Loss')
axes[0, 0].set_xlabel('Epoch')

# Individual components
for i, (ax, (name, vals)) in enumerate(zip(axes.flatten()[1:], loss_components.items())):
    ax.semilogy(vals)
    ax.set_title(name.replace('_', ' ').title())
    ax.set_xlabel('Epoch')

plt.suptitle('V8 Training Loss Components', fontsize=14)
plt.tight_layout()
plt.savefig('loss_curves_v8.png', dpi=150)
plt.show()

In [ ]:
#@title 1️⃣2️⃣ Save Model

save_dict = {
    'model_state_dict': model.state_dict(),
    'genes': genes,
    'metabolites': metabolites,
    'met_names': met_names,
    'env_vars': ENV_VARS,
    'stoichiometry': S,
    'gene_reaction_map': G,
    'training_losses': losses,
    'loss_components': loss_components,
    'knockout_results': ko_results,
    'version': 'v8_complete_integration',
    'features': [
        'trajectory_supervised',
        'hamiltonian_dynamics',
        'thermodynamic_loss',
        'conservation_loss',
        'km_diversity',
        'environment_coupling',
        'wreg_gene_regulation',
    ],
    'metrics': {
        'trajectory_corr': corr,
        'trajectory_mse': mse,
        'n_essential': len(essential),
        'km_std': float(Km.std()),
        'wreg_max': float(np.abs(W_reg).max()),
    }
}

torch.save(save_dict, 'dark_manifold_v8.pt')

print("="*60)
print("DARK MANIFOLD V8 - SUMMARY")
print("="*60)
print(f"\nFeatures integrated:")
for f in save_dict['features']:
    print(f"  ✅ {f}")

print(f"\nMetrics:")
print(f"  Trajectory correlation: {corr:.4f}")
print(f"  Trajectory MSE: {mse:.6f}")
print(f"  Essential genes: {len(essential)}/{n_genes}")
print(f"  Km diversity (std): {Km.std():.4f}")
print(f"  W_reg activity (max): {np.abs(W_reg).max():.4f}")

from google.colab import files
files.download('dark_manifold_v8.pt')
files.download('trajectory_v8.png')
files.download('environment_v8.png')
files.download('parameters_v8.png')
files.download('loss_curves_v8.png')

## V8 Summary

### What's Fixed

| Issue in V7 | Fix in V8 |
|-------------|----------|
| No trajectory supervision | ✅ ODE-generated targets + MSE loss |
| No Hamiltonian | ✅ Energy function + conservation loss |
| No W_reg | ✅ Gene regulation matrix + sparsity loss |
| No Km diversity | ✅ Diversity loss (std > 0.3) |
| Batch too large | ✅ Reduced to 8 |
| LR too high | ✅ Reduced to 3e-4 |

### Loss Components

```
total = 10.0 * trajectory_MSE      # Supervised ground truth
      +  1.0 * hamiltonian          # Energy conservation
      +  0.5 * thermodynamic        # ΔG × v ≤ 0
      +  0.5 * conservation         # Mass balance
      +  1.0 * energy_charge        # EC > 0.7
      +  2.0 * km_diversity         # Km std > 0.3
      +  1.0 * environment          # Glucose depletion
      +  1.0 * wreg_sparsity        # Gene regulation active
```

### Key Insight

The trajectory MSE provides the **supervised signal** that V7 was missing.
All other losses are **regularizers** that enforce physics, not replacements for supervision.